# Inspect FSD Cube muon selection

This notebook calls the production package without modifying simulation files.
Start in the production checkout or its `run-muon-skim` directory.
The path cell detects Casey's NERSC pilot; elsewhere, set source/output paths manually.
NumPy, h5py and PyYAML are required. For the interactive plot, install Plotly in
this notebook's kernel with `%pip install plotly nbformat` if needed.
The truth length cut is a preselection; reconstructed hit-count and shape cuts still apply.


In [ ]:
from pathlib import Path
import json
import sys
import h5py
import numpy as np

stage = Path.cwd()
if not (stage / 'muon_skim').is_dir():
    stage = stage / 'run-muon-skim'
if not (stage / 'muon_skim').is_dir():
    raise RuntimeError('Start this notebook in ND_Production or run-muon-skim')
sys.path.insert(0, str(stage.resolve()))
from muon_skim import load_config, cube_geometry, translations
from muon_skim.geometry import fingerprint
from muon_skim.io import read_tables, events
from muon_skim.selection import qualifying_translations, choose_translation


In [ ]:
import os

# NERSC pilot defaults. Outside NERSC, optional inspection stays disabled.
work_root = Path('/global/cfs/cdirs/dune/users/cborden')
job_root = work_root / 'muon-skim-jobs'
larnd_repo = stage.resolve().parents[1] / 'larnd-sim'
source_path = None
source_event_id = None
skim_directory = None
saved_record = None
config = load_config(stage / 'configs/fsd_cube.json')

if work_root.is_dir():
    larnd_repo = work_root / 'larnd-sim'
    source_path = Path((job_root / 'pilot_sources.txt').read_text().splitlines()[0])
    skim_directory = (
        Path(os.environ['SCRATCH']) / 'fsd-muon-production/output/run-muon-skim'
        / 'FSDCubeMuons_pilot_v1/EDEPSIM_H5/0000000'
        / 'FSDCubeMuons_pilot_v1.0000000'
    )
    manifest = json.loads((skim_directory / 'manifest.json').read_text())
    if manifest['status'] != 'complete':
        raise RuntimeError('Wait for the pilot skim to complete.')
    config = manifest['metadata']['config']
    if manifest['outputs']:
        with h5py.File(manifest['outputs'][0]['path'], 'r') as skim:
            saved_record = json.loads(skim['provenance/events'][0])
        source_event_id = saved_record['source_event_id']
    else:
        source_path = None
        print('Pilot completed with no selected events.')

# Manual overrides can go here. If inspecting another event, set saved_record = None.
# source_path = Path('/path/to/source.EDEPSIM.hdf5')
# source_event_id = 1
inspect_traj_id = None  # None chooses the first qualifying muon; set 8 for the example.


In [ ]:
bounds, geometry_sources = cube_geometry(larnd_repo, config['detector_config'])
shifts = translations(config)
print('Target bounds [cm], rows x/y/z:', bounds, sep='\n')
print('Strict minimum length [cm]:', config['min_length_cm'])
print('Candidate translations:', len(shifts))


In [ ]:
event = choice = None
if source_path is not None:
    info = fingerprint(source_path)
    with h5py.File(source_path, 'r') as source:
        for event_id, rows in events(read_tables(source)):
            if source_event_id is None or event_id == source_event_id:
                event = rows
                candidates = qualifying_translations(rows['segments'], shifts, bounds, config['min_length_cm'])
                choice = choose_translation(candidates, config['seed'], info['sha256'], event_id)
                print('Source event:', event_id)
                print('Qualifying placements:', len(candidates))
                print(json.dumps(choice, indent=2))
                break
        else:
            raise ValueError('Requested event not found')
else:
    print('Set source_path to inspect a source event.')


In [ ]:
from muon_skim.geometry import clip_segments
from muon_skim.selection import length_metrics

clipped_start = clipped_end = p0 = p1 = None
if choice is not None:
    if saved_record is not None:
        np.testing.assert_allclose(choice['shift_cm'], saved_record['shift_cm'], rtol=0, atol=0)
        print('Recomputed translation matches saved pilot translation.')

    tid = choice['muons'][0]['source_file_traj_id'] if inspect_traj_id is None else inspect_traj_id
    rows = event['segments']
    rows = rows[(np.abs(rows['pdg_id'].astype(np.int64)) == 13) & (rows['file_traj_id'] == tid)]
    starts = np.column_stack([rows[f'{a}_start'] for a in 'xyz']) + choice['shift_cm']
    ends = np.column_stack([rows[f'{a}_end'] for a in 'xyz']) + choice['shift_cm']
    clipped_start, clipped_end = clip_segments(starts, ends, bounds)
    if not len(clipped_start):
        raise ValueError(f'Muon {tid} has no deposited segments inside the Cube.')
    points = np.concatenate([clipped_start, clipped_end])
    centered = points - points.mean(axis=0)
    _, _, axes = np.linalg.svd(centered, full_matrices=False)
    projection = centered @ axes[0]
    p0, p1 = points[projection.argmin()], points[projection.argmax()]
    delta = p1 - p0
    chord, path = length_metrics(clipped_start, clipped_end)
    angle = np.degrees(np.arccos(np.clip(abs(delta[1]) / chord, 0, 1))) if chord else float('nan')
    print('Muon source_file_traj_id:', tid)
    print('Chord endpoints [x, y, z] cm:', p0, p1, sep='\n')
    print('Absolute displacement [dx, dy, dz] cm:', np.abs(delta))
    print('Chord and deposited path [cm]:', (chord, path))
    print('Chord angle from vertical [degrees]:', angle)
    print('Cube vertical height [cm]:', np.diff(bounds[1])[0])
    print('Deposits reach bottom / top:',
          bool(np.any(np.isclose(points[:, 1], bounds[1, 0], rtol=0, atol=1e-5))),
          bool(np.any(np.isclose(points[:, 1], bounds[1, 1], rtol=0, atol=1e-5))))
    print('Endpoints bound recorded deposits; an interior endpoint need not be a physical stopping point.')
else:
    print('No qualifying placement selected; endpoint inspection skipped.')


In [ ]:
# Drag to rotate; scroll to zoom; click legend entries to toggle traces.
# The plotted coordinates are x, z, y so the physical vertical axis stays vertical.
# This reconstructs the translated source deposits, not reconstructed detector hits.
fig = None
if choice is not None:
    try:
        import plotly.graph_objects as go
    except ImportError:
        print('Install Plotly in this kernel: %pip install plotly nbformat')
    else:
        import itertools
        fig = go.Figure()

        def add_segments(starts, ends, name, color, width=3, visible=True):
            # NaN separators prevent fictitious connections between deposits.
            xyz = np.full((len(starts), 3, 3), np.nan)
            xyz[:, 0], xyz[:, 1] = starts, ends
            xyz = xyz.reshape(-1, 3)
            fig.add_trace(go.Scatter3d(
                x=xyz[:, 0], y=xyz[:, 2], z=xyz[:, 1], mode='lines',
                name=name, line=dict(color=color, width=width), visible=visible,
                hovertemplate='x=%{x:.3f} cm<br>z=%{y:.3f} cm<br>y=%{z:.3f} cm<extra>%{fullData.name}</extra>',
            ))

        all_rows = event['segments']
        all_starts = np.column_stack([all_rows[f'{a}_start'] for a in 'xyz']) + choice['shift_cm']
        all_ends = np.column_stack([all_rows[f'{a}_end'] for a in 'xyz']) + choice['shift_cm']
        # Hidden initially to avoid obscuring the selected muon with the full event.
        add_segments(all_starts, all_ends, 'All event deposits (toggle)', 'lightgray', 2, 'legendonly')

        # Split the chosen muon's outside portions from the exact measured inside portions.
        outside_starts, outside_ends = [], []
        for start, end in zip(starts, ends):
            cs, ce = clip_segments(start[None, :], end[None, :], bounds)
            pieces = [(start, end)] if not len(cs) else [(start, cs[0]), (ce[0], end)]
            for a, b in pieces:
                if np.linalg.norm(b - a) > 1e-10:
                    outside_starts.append(a)
                    outside_ends.append(b)
        add_segments(np.asarray(outside_starts).reshape(-1, 3),
                     np.asarray(outside_ends).reshape(-1, 3), 'Selected muon: outside', 'gray', 3)
        add_segments(clipped_start, clipped_end, 'Selected muon: measured inside', 'royalblue', 6)
        # Optional chord is explicitly distinct from the actual deposit segments.
        add_segments(p0[None, :], p1[None, :], 'Endpoint chord (toggle)', 'orange', 4, 'legendonly')
        endpoints = np.stack([p0, p1])
        fig.add_trace(go.Scatter3d(
            x=endpoints[:, 0], y=endpoints[:, 2], z=endpoints[:, 1],
            mode='markers+text', text=['A', 'B'], name='Chord endpoints',
            marker=dict(size=6, color='crimson'),
            hovertemplate='x=%{x:.3f} cm<br>z=%{y:.3f} cm<br>y=%{z:.3f} cm<extra>Endpoint %{text}</extra>',
        ))
        corners = np.array(list(itertools.product(*bounds)))
        edges = [(a, b) for a, b in itertools.combinations(corners, 2) if np.count_nonzero(a != b) == 1]
        add_segments(np.array([a for a, b in edges]), np.array([b for a, b in edges]), 'Cube active boundary', 'black', 4)
        padding = np.diff(bounds, axis=1).ravel() * 0.08
        view_bounds = bounds + np.column_stack([-padding, padding])
        fig.update_layout(
            title=f'Translated source event {event_id}, muon {tid}: measured path {path:.3f} cm',
            height=750, margin=dict(l=0, r=0, b=0, t=55),
            scene=dict(
                xaxis=dict(title='x drift [cm]', range=view_bounds[0].tolist()),
                yaxis=dict(title='z transverse [cm]', range=view_bounds[2].tolist()),
                zaxis=dict(title='y vertical [cm]', range=view_bounds[1].tolist()),
                aspectmode='data', dragmode='orbit',
            ),
        )
        # Full-event view includes all deposits; Cube view restores the initial framing.
        fig.update_layout(updatemenus=[dict(type='buttons', buttons=[
            dict(label='Cube view', method='relayout', args=[{
                'scene.xaxis.range': view_bounds[0].tolist(), 'scene.xaxis.autorange': False,
                'scene.yaxis.range': view_bounds[2].tolist(), 'scene.yaxis.autorange': False,
                'scene.zaxis.range': view_bounds[1].tolist(), 'scene.zaxis.autorange': False}]),
            dict(label='Fit visible deposits', method='relayout', args=[{
                'scene.xaxis.autorange': True, 'scene.yaxis.autorange': True, 'scene.zaxis.autorange': True}]),
        ])])
        try:
            in_notebook = get_ipython() is not None
        except NameError:
            in_notebook = False
        if in_notebook:
            fig.show()
        # Optional standalone rotatable HTML, including JavaScript (no internet needed):
        # fig.write_html('muon-inspection.html', include_plotlyjs=True)


In [ ]:
if skim_directory is not None:
    manifest = json.loads((Path(skim_directory) / 'manifest.json').read_text())
    print({k: manifest[k] for k in ('status', 'scanned_events', 'selected_events')})
    for output in manifest['outputs'][:3]:
        with h5py.File(output['path']) as skim:
            print(output['path'], 'events:', output['events'])
            record = json.loads(skim['provenance/events'][0])
            print({k: v for k, v in record.items() if k != 'id_maps'})
            print('ID map columns:', skim['provenance'].attrs['id_map_columns_json'])


The original shower notebook is retained separately as a reference. Production runs use
`python -m muon_skim`, never this notebook. Geometry comes from the specified YAML checkout;
this plot is an active-volume schematic, not a new Geant4 transport calculation.
